# Power BI activity data ingestion

This notebook incrementally retrieves Power BI Admin Activity Events, backfills up to 27 days, removes duplicate events, and writes new activity records to the `pbi_activity_events` Delta table in the attached Lakehouse.

## Imports

In [ ]:
import json
import requests
from datetime import datetime, timedelta, timezone
import msal
from notebookutils import mssparkutils, runtime

from pyspark.sql import functions as F

## Define parameters

In [ ]:
TABLE_NAME = "pbi_activity_events"
POWERBI_RESOURCE = "https://analysis.windows.net/powerbi/api"
BASE_URL = "https://api.powerbi.com/v1.0/myorg/admin/activityevents"
MAX_DAYS_BACKFILL = 27 # The max possible is 27
WORKSPACE_FILTER = False

vaulturl = "https://<your-key-vault-name>.vault.azure.net/"
TENANT_ID = "<your-tenant-id-guid>"


## Helpers

In [ ]:
def fetch_all_activity_events(start_iso: str, end_iso: str, activities: list[str] | None = None):

    # If no filter → call API once
    if not activities:
        activities = [None]

    all_events = []
    for activity in activities:
        url = f"{BASE_URL}?startDateTime='{start_iso}'&endDateTime='{end_iso}'"

        if activity:
            url += f"&$filter=Activity eq '{activity}'"
        while True:
            resp = requests.get(url, headers=headers, timeout=120)
            resp.raise_for_status()
            payload = resp.json()

            events = payload.get("activityEventEntities", []) or []
            all_events.extend(events)

            cont = payload.get("continuationUri")
            last = payload.get("lastResultSet")

            if last is True or not cont:
                break

            url = cont

    return all_events

## Authenticate

In [ ]:

CLIENT_ID = mssparkutils.credentials.getSecret(vaulturl, "usageclient")
CLIENT_SECRET = mssparkutils.credentials.getSecret(vaulturl, "usagesecret")

if not (TENANT_ID and CLIENT_ID and CLIENT_SECRET):
    raise RuntimeError(
        "No token available. Either enable mssparkutils token acquisition, "
        "or set TENANT_ID/CLIENT_ID/CLIENT_SECRET env vars for MSAL."
    )

authority = f"https://login.microsoftonline.com/{TENANT_ID}"
app = msal.ConfidentialClientApplication(
    client_id=CLIENT_ID,
    authority=authority,
    client_credential=CLIENT_SECRET
)
# Use .default for the Power BI resource
scopes = [f"{POWERBI_RESOURCE}/.default"]
result = app.acquire_token_for_client(scopes=scopes)
if "access_token" not in result:
    raise RuntimeError(f"Failed to acquire token via MSAL: {result}")
else:
    print("Authentication succesfull using service principal")
token = result["access_token"]

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

## Get list of dates to fetch

In [ ]:
table_exists = spark.catalog.tableExists(TABLE_NAME)

if table_exists:
    max_time = spark.sql(f"""
        SELECT max(CAST(CreationTime AS TIMESTAMP)) AS max_ct
        FROM {TABLE_NAME}
    """).first()["max_ct"]

    last_loaded_date = max_time.date() if max_time is not None else None
else:
    last_loaded_date = None

today_utc = datetime.now(timezone.utc).date()
yesterday = today_utc - timedelta(days=1)

dates_to_fetch = []

if last_loaded_date is None:
    start_date = yesterday - timedelta(days=MAX_DAYS_BACKFILL - 1)
else:
    start_date = last_loaded_date + timedelta(days=1)

min_allowed = yesterday - timedelta(days=MAX_DAYS_BACKFILL - 1)
if start_date < min_allowed:
    start_date = min_allowed

current = start_date
while current <= yesterday:
    dates_to_fetch.append(current)
    current += timedelta(days=1)

print(f"Fetching {len(dates_to_fetch)} days:", dates_to_fetch)

## Fetch all days

In [ ]:
all_events = []
for day in dates_to_fetch:
    start_dt = datetime(day.year, day.month, day.day, 0, 0, 0, tzinfo=timezone.utc)
    end_dt   = datetime(day.year, day.month, day.day, 23, 59, 59, tzinfo=timezone.utc)

    start_str = start_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    end_str   = end_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

    print(f"Fetching events for {day}")
    activities = ['ViewReport', 
    'DeleteReport', 'CreateReport', 'UpdateReportContent', 'EditReport', 
    'DownloadReport', 'ExportReport', 'PrintReport', 'CopyReport', 'ShareReport', 'AnalyzeInExcel']

    events = fetch_all_activity_events(start_str, end_str, activities)
    print(f"  -> {len(events)} events")

    all_events.extend(events)

print(f"Total events fetched: {len(all_events)}")

## write events to lakehouse

In [ ]:
if len(all_events) == 0:
    print("No events returned. Nothing to append.")

else:
    rdd = spark.sparkContext.parallelize([json.dumps(e) for e in all_events])
    df = spark.read.json(rdd)

    df = df.withColumn("ingestion_date_utc", F.current_date())

    needed_cols = ["Activity", "AppId", "AppName", "ArtifactKind", "CapacityId", "CapacityName", "DatasetId", "DatasetName", "ReportId", "ReportName", "WorkspaceId", "WorkSpaceName", "UserId", "UserType", "UserKey", "CreationTime", "Id"]
    columns_to_select = [c for c in needed_cols if c in df.columns]
    missing_cols = list(set(needed_cols)-set(columns_to_select))
    df = df[columns_to_select]
    if len(missing_cols) > 0:
      print(f"Missing columns: {missing_cols}")

    if WORKSPACE_FILTER and "WorkspaceId" in df.columns:
      workspace_id = runtime.context.get("currentWorkspaceId")
      print(f"Filtering for workspace: {workspace_id}")
      df = df[df['WorkspaceId'] == workspace_id]

    if not table_exists:
        print("Table does not exist → creating table")

        (df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(TABLE_NAME))

    else:
        existing_ids = spark.read.table(TABLE_NAME).select("Id")
        len_before = df.count()
        df = df.join(existing_ids, on="Id", how="left_anti")
        len_after = df.count()
        print(f"removed {len_before-len_after} duplicate rows")

        if len_after == 0:
          print("No new rows found, nothing to append")
        else:
          (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(TABLE_NAME))

          print(f"Appended {len_after} rows to {TABLE_NAME}")